# FirstAI — integrated build through Step 3
Run the cells from top to bottom. This notebook verifies the code, checks the full training pipeline on the attached GPU, then provides the persistent real-training workflow. It does not define identity, personality or conversational behaviour.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('Real T4 training is blocked because CUDA is unavailable. Fix the runtime before training.')
if not Path(CORPUS).is_file():
    raise RuntimeError(f'Corpus file not found: {CORPUS}. Upload and prepare your .txt corpus first.')
subprocess.run([
    sys.executable, 'train.py', '--data', CORPUS, '--out', CHECKPOINT,
    '--steps', '1000', '--eval-interval', '100', '--checkpoint-interval', '250',
    '--batch-size', '16', '--block-size', '128', '--layers', '4', '--heads', '4',
    '--width', '128', '--grad-accum-steps', '2', '--amp'
], check=True)

## Test 1 — automated unit tests
Checks tokenizer, corpus preparation, data batching, model shape, causal attention and gradients.

In [ ]:
!python -m unittest discover -s tests -v

## Test 2 — training engine and checkpoint recovery
Runs a tiny temporary training job, saves latest/best checkpoints, resumes, restores scaler/RNG state, evaluates and generates. This is an engineering test, not meaningful knowledge training.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        'Test 2 needs a CUDA-enabled Colab runtime. The current PyTorch build is '
        f'{torch.__version__} (CUDA build: {torch.version.cuda}). '
        'Select T4 GPU, reconnect the runtime, and rerun the setup cell.'
    )
!python scripts/smoke_test.py --require-cuda

## Test 3 — integrated GPU pipeline with persistent storage
Mount Google Drive so real training data and checkpoints survive a Colab disconnect.

In [ ]:
# Drive is optional for running tests. If authorization fails, continue safely in /content.
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    drive_root = Path('/content/drive/MyDrive')
    if not drive_root.is_dir():
        raise RuntimeError('Google Drive mounted but MyDrive is not accessible.')
    BASE = drive_root / 'FirstAI'
    print('Persistent Google Drive storage enabled.')
except Exception as exc:
    print('\nGoogle Drive mount failed:', repr(exc))
    print('Continuing with temporary Colab storage. Files will be lost when this runtime disconnects.')
    BASE = LOCAL_BASE

RAW = BASE / 'data' / 'raw'
OUT = BASE / 'checkpoints'
CORPUS = str(BASE / 'data' / 'input.txt')
CHECKPOINT = str(OUT / 'firstgpt.pt')
RAW.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
print('Workspace:', BASE)
print('Raw corpus folder:', RAW)
print('Checkpoint folder:', OUT)

### Add your training corpus
Upload UTF-8 `.txt` files you have permission to use. They are stored in your private Google Drive folder, not committed to the public GitHub repository.

In [ ]:
from google.colab import files
uploaded = files.upload()
txt_files = [(name, data) for name, data in uploaded.items() if name.lower().endswith('.txt')]
if not txt_files:
    print('No .txt files selected. This is not a code failure; upload UTF-8 .txt files and rerun this cell.')
else:
    for name, data in txt_files:
        target = RAW / Path(name).name
        target.write_bytes(data)
    print('Saved', len(txt_files), 'text file(s) to', RAW)

In [ ]:
if not any(p.is_file() and p.suffix.lower() == '.txt' for p in RAW.rglob('*')):
    raise RuntimeError(f'No .txt corpus files found under {RAW}. Run the upload cell and select at least one .txt file.')
(BASE / 'data').mkdir(parents=True, exist_ok=True)
subprocess.run([
    sys.executable, 'prepare_corpus.py',
    '--input-dir', str(RAW), '--output', CORPUS
], check=True)
if not Path(CORPUS).is_file():
    raise RuntimeError(f'Corpus preparation did not create {CORPUS}')
print('Corpus ready:', CORPUS, '| bytes:', Path(CORPUS).stat().st_size)

### Train from random initialization
Starter configuration for a T4. This is a small baseline, not a claim that the model will become a capable assistant. Training can be resumed using the next cell.

In [ ]:
if not Path(CHECKPOINT).is_file():
    raise RuntimeError(f'No resumable checkpoint found at {CHECKPOINT}. Run the initial training cell first.')
subprocess.run([
    sys.executable, 'train.py', '--data', CORPUS, '--out', CHECKPOINT,
    '--resume', '--steps', '1000', '--eval-interval', '100',
    '--checkpoint-interval', '250', '--batch-size', '16',
    '--grad-accum-steps', '2', '--amp'
], check=True)

### Resume training if needed
Run this cell only after a previous training run has created the checkpoint. It uses the same corpus and restores the saved model, optimizer, mixed-precision scaler and RNG states.

In [ ]:
!python train.py --data "$CORPUS" --out "$CHECKPOINT" --resume --steps 1000 --eval-interval 100 --checkpoint-interval 250 --batch-size 16 --grad-accum-steps 2 --amp

### Evaluate and generate
Evaluation checks the exact corpus fingerprint. Generation uses the best validation checkpoint.

In [ ]:
BEST_CHECKPOINT = str(OUT / 'firstgpt.best.pt')
if not Path(BEST_CHECKPOINT).is_file():
    raise RuntimeError(f'Best checkpoint not found: {BEST_CHECKPOINT}. Training must finish successfully first.')
subprocess.run([
    sys.executable, 'evaluate.py', '--checkpoint', BEST_CHECKPOINT,
    '--data', CORPUS, '--batches', '50', '--batch-size', '16'
], check=True)
subprocess.run([
    sys.executable, 'generate.py', '--checkpoint', BEST_CHECKPOINT,
    '--prompt', 'The ', '--tokens', '200', '--temperature', '0.8', '--top-k', '40'
], check=True)

### Download a copy of the checkpoints
The originals remain in Drive. Download these copies if you also want them on your phone.

In [ ]:
if not Path(CHECKPOINT).is_file() or not Path(BEST_CHECKPOINT).is_file():
    raise RuntimeError('Both latest and best checkpoints must exist before downloading.')
files.download(CHECKPOINT)
files.download(BEST_CHECKPOINT)